In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import (
    train_test_split,GridSearchCV,cross_val_predict)

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)
from lightgbm import LGBMClassifier


df=pd.read_csv("../data/ai4i2020.csv")

df=df.drop(columns=["UDI","Product ID"])

df["Power"]=(
    df["Torque [Nm]"]*
    df["Rotational speed [rpm]"]
)


type_mapping={
    "L":0,
    "M":1,
    "H":2
}

df["Type"]=df["Type"].map(type_mapping)

target="TWF"

failure_columns=[
    "Machine failure",
    "HDF",
    "PWF",
    "TWF",
    "OSF",
    "RNF"
]

X=df.drop(columns=failure_columns)
y=df[target]

X.columns = (
    X.columns
    .str.replace("[", "", regex=False)
    .str.replace("]", "", regex=False)
    .str.replace("<", "", regex=False)
)

X_train,X_test,y_train,y_test=train_test_split(
    X,y,
    test_size=0.3,
    stratify=y
)


model=LGBMClassifier(
    objective="binary",
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)


param_grid={
    "n_estimators":[100,200,300],
    "max_depth":[-1,3,5],
    "learning_rate":[0.03,0.05,0.1],
    "num_leaves":[15,31,63],
    "subsample":[0.8,1.0],
    "colsample_bytree":[0.8,1.0]
}

grid=GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    verbose=2
)

grid.fit(X_train,y_train)

best_model=grid.best_estimator_

cv_probs=cross_val_predict(
    best_model,
    X_train,
    y_train,
    cv=5,
    method="predict_proba",
    n_jobs=-1
)[:,1]

print("Minimum probability:",cv_probs.min())
print("Maximum probability:",cv_probs.max())
print("Mean probability:",cv_probs.mean())


thresholds=np.arange(0.01,1.0,0.01)

best_threshold=0
best_f1=0

for threshold in thresholds:
    preds=(cv_probs>=threshold).astype(int)

    precision = precision_score(
        y_train,
        preds,
        zero_division=0
    )

    recall = recall_score(
        y_train,
        preds,
        zero_division=0
    )

    f1 = f1_score(
        y_train,
        preds,
        zero_division=0
    )

    tn,fp,fn,tp=confusion_matrix(
        y_train,
        preds,
        labels=[0,1]
    ).ravel()

    print(
        f"{threshold:.2f}"
        f"{precision:.4f}"
        f"{recall:.4f}"
        f"{f1:.4f}"
        f"{fp}"
        f"{fn}"
    )

    if f1>best_f1:
        best_f1=f1
        best_threshold=threshold

print("\nSelected threshold:",best_threshold)
print("Best OOF F1:",best_f1)

Fitting 5 folds for each of 324 candidates, totalling 1620 fits
Minimum probability: 0.00022018367701084828
Maximum probability: 0.5300219562589107
Mean probability: 0.00238173329433976
0.010.06670.46880.116721017
0.020.05560.25000.090913624
0.030.05310.18750.082810726
0.040.07140.18750.10347826
0.050.07790.18750.11017126
0.060.08960.18750.12126126
0.070.08930.15620.11365127
0.080.07140.09380.08113929
0.090.07320.09380.08223829
0.100.08820.09380.09093129
0.110.10000.09380.09682729
0.120.07690.06250.06902430
0.130.04350.03120.03642231
0.140.00000.00000.00002032
0.150.00000.00000.00001832
0.160.00000.00000.00001632
0.170.00000.00000.00001532
0.180.00000.00000.00001232
0.190.00000.00000.00001232
0.200.00000.00000.00001232
0.210.00000.00000.00001232
0.220.00000.00000.00001232
0.230.00000.00000.00001232
0.240.00000.00000.00001232
0.250.00000.00000.00001132
0.260.00000.00000.00001132
0.270.00000.00000.00001132
0.280.00000.00000.00001032
0.290.00000.00000.0000932
0.300.00000.00000.0000932
0.3

In [4]:
best_model = grid.best_estimator_

test_probs = best_model.predict_proba(X_test)[:, 1]

threshold = 0.06

test_preds = (
    test_probs >= threshold
).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_test,
    test_preds,
    labels=[0, 1]
).ravel()

accuracy = accuracy_score(y_test, test_preds)
precision = precision_score(y_test, test_preds, zero_division=0)
recall = recall_score(y_test, test_preds, zero_division=0)
f1 = f1_score(y_test, test_preds, zero_division=0)
roc_auc = roc_auc_score(y_test, test_probs)
pr_auc = average_precision_score(y_test, test_probs)

print("=" * 60)
print("LIGHTGBM — TWF — OOF SELECTED THRESHOLD")
print("=" * 60)

print(f"Threshold: {threshold:.2f}")

print("\nConfusion matrix:")
print(f"TN: {tn}")
print(f"FP: {fp}")
print(f"FN: {fn}")
print(f"TP: {tp}")

print("\nTest metrics:")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1       : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")
print(f"PR-AUC   : {pr_auc:.4f}")

print("=" * 60)

LIGHTGBM — TWF — OOF SELECTED THRESHOLD
Threshold: 0.06

Confusion matrix:
TN: 2954
FP: 32
FN: 13
TP: 1

Test metrics:
Accuracy : 0.9850
Precision: 0.0303
Recall   : 0.0714
F1       : 0.0426
ROC-AUC  : 0.9565
PR-AUC   : 0.0589
